**SISTEMATIZAÇÃO DE CIÊNCIA DE DADOS II**

Aluna: Thamires Gil Godoy

Dados: Top Spotify Podcast Episodes

Dados disponiveis em: https://www.kaggle.com/datasets/daniilmiheev/top-spotify-podcasts-daily-updated


*   date: data de quando os dados foram coletados
*   rank: a posição diária de um podcast entre os 200 primeiros
*   region: o código ISO do país
*   chart rank move: a variação na classificação com relação ao dia anterior
*   episodeURI: identificador único do episódio no Spotify
*   showURI: identificador único do programa no Spotify
*   episodeName: Nome do episódio
*   description: Descrição do episódio. Alguns episódios não possuem.
*   show.name: Nome do programa
*   show.publisher: Descrição do programa
*   duration_ms: duração do episodio em ms
*   explicit: boleano definindo se os dados são explicitos ou não
*   languages: idioma
*   release_date: data de lançamento
*   show.media_type: forma de midia diponivel (audio, video)
*   show.total_episodes: total de episodios

0. PREPARAÇÃO DO AMBIENTE

In [ ]:
# Instalações necessárias
!pip install kaggle -q

In [ ]:
# Bibliotecas necessárias
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Sistematizacao") \
    .getOrCreate()

from pyspark.sql.functions import col, sum, count, round, to_date, expr, when, avg, first, date_format
from pyspark.sql.window import Window
from collections import Counter
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

1. INGESTÃO E PRÉ-PROCESSAMENTO

In [ ]:
# Primeiro os dados são baixados
!kaggle datasets download -d daniilmiheev/top-spotify-podcasts-daily-updated -p ./data --unzip

Dataset URL: https://www.kaggle.com/datasets/daniilmiheev/top-spotify-podcasts-daily-updated
License(s): ODC Attribution License (ODC-By)
100% 645M/645M [00:08<00:00, 76.2MB/s]



In [ ]:
# Em seguida, são lidos e alguns são mostrados
# Nesse ponto, a visualização de 15 ou 20 registros não mostrava um erro que foi percebido ao longo da limpeza:
# a locação de dados em colunas erradas
# com 50 isso já é verificado
df_spotify_csv = spark.read.csv ("/content/data/top_podcasts.csv",header=True)
df_spotify_csv.show(50)

# guardar a qunatidade de registros antes da limpeza
qtd_linhas_antes = df_spotify_csv.count()

+----------+----+------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------------+------------+---------------+-------------------+
|      date|rank|region|chartRankMove|          episodeUri|             showUri|         episodeName|         description|           show.name|      show.publisher|         duration_ms|            explicit|       languages|release_date|show.media_type|show.total_episodes|
+----------+----+------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------------+------------+---------------+-------------------+
|2024-09-02|   1|    us|    UNCHANGED|37kBZRr3nqjltQXt8...|4rOoJ6Egrf8K2Iryw...|  #2197 - Mike Baker|"Mike Baker is a ...|The Joe Rogan Exp...|           Joe Rogan|           987210

In [ ]:
# Para melhorar o entendimento e para que não ocorra erro devido a nomes com ponto, as colunas são renomeadas

df_spotify_csv = df_spotify_csv.withColumnRenamed("date", "data")
df_spotify_csv = df_spotify_csv.withColumnRenamed("rank", "posicao")
df_spotify_csv = df_spotify_csv.withColumnRenamed("region", "pais")
df_spotify_csv = df_spotify_csv.withColumnRenamed("chartRankMove", "var_classifica")
df_spotify_csv = df_spotify_csv.withColumnRenamed("episodeUri", "id_episodio")
df_spotify_csv = df_spotify_csv.withColumnRenamed("showUri", "id_programa")
df_spotify_csv = df_spotify_csv.withColumnRenamed("episodeName", "nome_episodio")
df_spotify_csv = df_spotify_csv.withColumnRenamed("description", "descricao")
df_spotify_csv = df_spotify_csv.withColumnRenamed("show.name", "nome_programa")
df_spotify_csv = df_spotify_csv.withColumnRenamed("show.publisher", "editora")
df_spotify_csv = df_spotify_csv.withColumnRenamed("duration_ms", "duracao_ms")
df_spotify_csv = df_spotify_csv.withColumnRenamed("explicit", "explicito")
df_spotify_csv = df_spotify_csv.withColumnRenamed("languages", "idioma")
df_spotify_csv = df_spotify_csv.withColumnRenamed("release_date", "data_lancamento")
df_spotify_csv = df_spotify_csv.withColumnRenamed("show.media_type", "forma_midia")
df_spotify_csv = df_spotify_csv.withColumnRenamed("show.total_episodes", "epis_totais")

df_spotify_csv.show(8)

+----------+-------+----+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+---------+---------+---------------+-----------+-----------+
|      data|posicao|pais|var_classifica|         id_episodio|         id_programa|       nome_episodio|           descricao|       nome_programa|             editora|duracao_ms|explicito|   idioma|data_lancamento|forma_midia|epis_totais|
+----------+-------+----+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+---------+---------+---------------+-----------+-----------+
|2024-09-02|      1|  us|     UNCHANGED|37kBZRr3nqjltQXt8...|4rOoJ6Egrf8K2Iryw...|  #2197 - Mike Baker|"Mike Baker is a ...|The Joe Rogan Exp...|           Joe Rogan| 9872105.0|     True|   ['en']|     2024-08-29|      mixed|     2366.0|
|2024-09-02|      2|  us|     UNCHANGED|293KKxbE

In [ ]:
# Para esse trabalho algumas colunas não serão usadas, por isso serão excluidas

df_spotify_csv = df_spotify_csv.drop("id_episodio", "id_programa")
df_spotify_csv.printSchema() #e pelo shcema verifica-se que os tipos de dados não estão adequados
df_spotify_csv.show(15)


root
 |-- data: string (nullable = true)
 |-- posicao: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- var_classifica: string (nullable = true)
 |-- nome_episodio: string (nullable = true)
 |-- descricao: string (nullable = true)
 |-- nome_programa: string (nullable = true)
 |-- editora: string (nullable = true)
 |-- duracao_ms: string (nullable = true)
 |-- explicito: string (nullable = true)
 |-- idioma: string (nullable = true)
 |-- data_lancamento: string (nullable = true)
 |-- forma_midia: string (nullable = true)
 |-- epis_totais: string (nullable = true)

+----------+-------+----+--------------+--------------------+--------------------+--------------------+--------------------+----------+---------+---------+---------------+-----------+-----------+
|      data|posicao|pais|var_classifica|       nome_episodio|           descricao|       nome_programa|             editora|duracao_ms|explicito|   idioma|data_lancamento|forma_midia|epis_totais|
+----------+-------+-

In [ ]:
# O próximo passo foi analisar a porcentagem de nulos que existem

#total de linhas
t_linhas = df_spotify_csv.count()

#porcentagem de nulos por coluna
porc_nulos = df_spotify_csv.select([
    (sum(col(c).isNull().cast("int")) / t_linhas).alias(c)
    for c in df_spotify_csv.columns
])

porc_nulos.show()


# O resultado mostra que a coluna descricao é a que possui maior porcentagem de nulos (0,00873%), mas nenhuma coluna chega a 1%
# Aparentemente os valores nulos parecem não possuir grande influencia no conjunto de dados,
# no entanto, verificou-se que certas colunas resultaram em valores similares e assim, a análise dos valores nulos seguiu

+----+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|data|             posicao|                pais|      var_classifica|       nome_episodio|          descricao|       nome_programa|             editora|          duracao_ms|           explicito|              idioma|     data_lancamento|         forma_midia|         epis_totais|
+----+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
| 0.0|1.161758117603322E-5|1.161758117603322E-5|1.161758117603322E-5|1.161758117603322E-5|0.00873206445143597|1.909639905810460...|2.127469552861083...|1.967727811

In [ ]:
# Pelo resultado anterior, temos que a porcentagem de dados nulos em posicao, pais, var_classifica e nome_episodio é a mesma
# Dessa forma, questinou-se se o registro que é nulo em uma dessas colunas também está nulo nas outras
# Assim, contou-se a quantidade de registros nulos ao mesmo tempo nessas quatro colunas e obteve-se a porcetagem com relação ao total

# Linhas onde todas as colunas especificadas são nulas
linhas_nulas = df_spotify_csv.filter(
    col("posicao").isNull() &
    col("pais").isNull() &
    col("var_classifica").isNull() &
    col("nome_episodio").isNull()
)

# Porcetagem de registros nulos nas quatro colunas ao mesmo tempo
qtd_nulas = linhas_nulas.count()
porc_nulos_comuns = qtd_nulas/t_linhas
print(qtd_nulas)
print(porc_nulos_comuns)

# Assim descobrimos que todos os registros nulos em uma dessas colunas também está nulo nas outras 3, portanto, esses 16 registros foram excluidos
df_spotify_trat = df_spotify_csv.na.drop(subset=["posicao", "pais", "var_classifica", "nome_episodio"])

16
1.161758117603322e-05


In [ ]:
# O tratamento dos outros valores nulos será o seguinte:
# descricao: preencher com "nao informado"
# nome_programa: preencher com "nao informado"
# editora: preencher com "nao informado"
# duracao_ms: colocar a media dos outros registros do mesmo programa
# explicito: excluir registros nulos
# idioma: copiar de outro registro do mesmo programa (quando disponível)
# data_lancamento: excluir registros nulos
# forma_midia: excluir registros nulos
# epis_totais: copiar de outro registro do mesmo programa (quando disponível)


df_spotify_trat = df_spotify_trat.filter(
    col("explicito").isNotNull() &
    col("data_lancamento").isNotNull() &
    col("forma_midia").isNotNull())

df_spotify_trat = df_spotify_trat.fillna({
    "descricao": "nao informado",
    "nome_programa": "nao informado",
    "editora": "nao informado"})


In [ ]:
# A partir daqui o tratamento de valores nulos é pausado, pois tratar duracao_ms,
# idioma e epis totais gerara erros com os registros embaralhados

# A verificação de registros duplicados (escritos de forma diferentes) será realizada para as colunas:
# pais, nome_programa e idiomas

# Para pais:
df_spotify_trat.select("pais").distinct().show(200, truncate=False)
#Como verificado, todos os códigos dos países estão corretos, não demonstrando nenhum erro de incompatibilidade a ISO

+----+
|pais|
+----+
|us  |
|cl  |
|jp  |
|pl  |
|in  |
|au  |
|gb  |
|co  |
|de  |
|br  |
|es  |
|it  |
|ar  |
|ph  |
|nl  |
|ca  |
|nz  |
|at  |
|mx  |
|fr  |
|ie  |
|id  |
+----+



In [ ]:
# Para idioma

df_spotify_trat.select("idioma").distinct().show(200, truncate=False)
# A partir desse ponto, se observou que desde o carregamento dos dados há registros com dados trocados
# A solução adotada: exclui-los


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|idioma                                                                                                                

In [ ]:
# Para exclui-los utilizou-se do principio que todo registro de idioma deve estar entre [ ]
df_spotify_trat = df_spotify_trat.filter(
    col("idioma").startswith("[") & col("idioma").endswith("]")
)

In [ ]:
df_spotify_trat.select("nome_programa").distinct().show(1000, truncate=False)
#Como verificado, os nomes dos programas aparentam não possuir erro de nomeção
# erro que havia antes de excluir os dados embaralhados


+-----------------------------------------------------------------------------------+
|nome_programa                                                                      |
+-----------------------------------------------------------------------------------+
|Lex Fridman Podcast                                                                |
|Stuff You Should Know                                                              |
|Between Two Beers Podcast                                                          |
|Just Married: The Anthea Bradshaw Mystery                                          |
|Las Damitas Histeria                                                               |
|となりの雑談                                                                       |
|Hapa英会話 Podcast                                                                 |
|Detective Vikrant                                                                  |
|Review Buku Bisnis, Keuangan, Kehidupan.                      

In [ ]:
# Verificando quantos registros foram perdidos até então

qtd_linhas_depois = df_spotify_trat.count()

print("Quantidade de linhas antes da limpeza:", qtd_linhas_antes)
print("Quantidade de linhas depois da limpeza:", qtd_linhas_depois)
porc_linhas_removidas =  ((qtd_linhas_antes-qtd_linhas_depois)*100) / qtd_linhas_antes
print("Porcentagem de linhas removidas:", porc_linhas_removidas,"%")


Quantidade de linhas antes da limpeza: 1377223
Quantidade de linhas depois da limpeza: 1238021
Porcentagem de linhas removidas: 10.107440842913602 %


In [ ]:
# Agora vamos tratar os nulos que faltam
# E ao verificar qual coluna ainda possui nulos, só epis_totais tem 964 registros

qtd_nulos = df_spotify_trat.select([
    (sum(col(c).isNull().cast("int"))).alias(c)
    for c in df_spotify_trat.columns
])

qtd_nulos.show()

# Então não há a necessidade de:
# duracao_ms: colocar a media dos outros registros do mesmo programa
# idioma: copiar de outro registro do mesmo programa (quando disponível)
# Mas há a necessidade de:
# epis_totais: copiar de outro registro do mesmo programa (quando disponível)

+----+-------+----+--------------+-------------+---------+-------------+-------+----------+---------+------+---------------+-----------+-----------+
|data|posicao|pais|var_classifica|nome_episodio|descricao|nome_programa|editora|duracao_ms|explicito|idioma|data_lancamento|forma_midia|epis_totais|
+----+-------+----+--------------+-------------+---------+-------------+-------+----------+---------+------+---------------+-----------+-----------+
|   0|      0|   0|             0|            0|        0|            0|      0|         0|        0|     0|              0|          0|        964|
+----+-------+----+--------------+-------------+---------+-------------+-------+----------+---------+------+---------------+-----------+-----------+



In [ ]:
# Primeiro, agrupar por programa
w = Window.partitionBy("nome_programa")

# Depois preencho epis_totais nulo com o primeiro valor não nulo do mesmo programa
df_spotify_trat = df_spotify_trat.withColumn(
    "epis_totais",
    when(col("epis_totais").isNull(), first("epis_totais", ignorenulls=True).over(w))
    .otherwise(col("epis_totais"))
)

In [ ]:
# Verificando os nulos que ainda existem, temos só epis_totais com 1 registro nulo. Solução: excluir

qtd_nulos = df_spotify_trat.select([
    (sum(col(c).isNull().cast("int"))).alias(c)
    for c in df_spotify_trat.columns
])

qtd_nulos.show()

df_spotify_trat = df_spotify_trat.filter(col("epis_totais").isNotNull())

+----+-------+----+--------------+-------------+---------+-------------+-------+----------+---------+------+---------------+-----------+-----------+
|data|posicao|pais|var_classifica|nome_episodio|descricao|nome_programa|editora|duracao_ms|explicito|idioma|data_lancamento|forma_midia|epis_totais|
+----+-------+----+--------------+-------------+---------+-------------+-------+----------+---------+------+---------------+-----------+-----------+
|   0|      0|   0|             0|            0|        0|            0|      0|         0|        0|     0|              0|          0|          1|
+----+-------+----+--------------+-------------+---------+-------------+-------+----------+---------+------+---------------+-----------+-----------+



In [ ]:
qtd_linhas_depois = df_spotify_trat.count()

print("Quantidade de linhas antes da limpeza:", qtd_linhas_antes)
print("Quantidade de linhas depois da limpeza:", qtd_linhas_depois)
porc_linhas_removidas =  ((qtd_linhas_antes-qtd_linhas_depois)*100) / qtd_linhas_antes
print("Porcentagem de linhas removidas:", porc_linhas_removidas,"%")


Quantidade de linhas antes da limpeza: 1377223
Quantidade de linhas depois da limpeza: 1238020
Porcentagem de linhas removidas: 10.107513452795953 %


In [ ]:
# Como verificado anteriormente, os tipos de dados não estão condizente. Portanto, houve a transformação dos tipos de dados.
# data: string (nullable = true) --> date
# duracao_ms: string (nullable = true) --> float
# explicito: string (nullable = true) --> boleano
# data_lancamento: string (nullable = true) --> date
# epis_totais: string (nullable = true) --> float --> int (dessa forma não gera NULL)
df_spotify_trans = (
    df_spotify_trat
    .withColumn("data", to_date(col("data"), "yyyy-MM-dd"))
    .withColumn("duracao_ms", expr("try_cast(duracao_ms as float)"))
    .withColumn("explicito", expr("try_cast(explicito as boolean)"))
    .withColumn("data_lancamento", to_date(col("data_lancamento"), "yyyy-MM-dd"))
    .withColumn("epis_totais", expr("try_cast(epis_totais as float)"))
)
df_spotify_trans = df_spotify_trans.withColumn(
    "epis_totais", col("epis_totais").cast("int")
)


df_spotify_trans.printSchema()
df_spotify_trans.show(50)

# Ao transformar epis_totais de string para inteiro, o erro seguinte aparece:
# NumberFormatException: [CAST_INVALID_INPUT] The value '2366.0' of the type "STRING" cannot be cast to "INT" because it is malformed
# Uma solução é usar o try_cast, que coloca NULL nos dados que não puderam ser transformados
# No entanto, para esses dados não se perderem e como não há previsão de calculos para epis_total, a transformaÇão de type não foi feita

root
 |-- data: date (nullable = true)
 |-- posicao: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- var_classifica: string (nullable = true)
 |-- nome_episodio: string (nullable = true)
 |-- descricao: string (nullable = false)
 |-- nome_programa: string (nullable = false)
 |-- editora: string (nullable = false)
 |-- duracao_ms: float (nullable = true)
 |-- explicito: boolean (nullable = true)
 |-- idioma: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- forma_midia: string (nullable = true)
 |-- epis_totais: integer (nullable = true)

+----------+-------+----+--------------+--------------------+--------------------+--------------------+--------------------+----------+---------+------+---------------+-----------+-----------+
|      data|posicao|pais|var_classifica|       nome_episodio|           descricao|       nome_programa|             editora|duracao_ms|explicito|idioma|data_lancamento|forma_midia|epis_totais|
+----------+-------+----+--

In [ ]:
# Para auxiliar na análise se criou uma nova coluna que classifica a duracao de um episodio como longo, medio ou curto
df_spotify_eng = df_spotify_trans.withColumn(
    "faixa_duracao",
    when((col("duracao_ms")/60000) < 15, "curto")
    .when((col("duracao_ms")/60000).between(15, 45), "medio")
    .otherwise("longo")
)

+----------+-------+----+--------------+--------------------+--------------------+-------------+-------------+----------+---------+------+---------------+-----------+-----------+-------------+
|      data|posicao|pais|var_classifica|       nome_episodio|           descricao|nome_programa|      editora|duracao_ms|explicito|idioma|data_lancamento|forma_midia|epis_totais|faixa_duracao|
+----------+-------+----+--------------+--------------------+--------------------+-------------+-------------+----------+---------+------+---------------+-----------+-----------+-------------+
|2025-10-30|    119|  pl|           NEW|Czy Polska Oddała...|W tym odcinku ser...| Dawid Sołtan| Dawid Sołtan| 3239658.0|    false|['pl']|     2025-10-28|      mixed|         85|        longo|
|2025-10-31|    187|  pl|          DOWN|Czy Polska Oddała...|W tym odcinku ser...| Dawid Sołtan| Dawid Sołtan| 3239658.0|    false|['pl']|     2025-10-28|      mixed|         85|        longo|
|2025-11-10|     95|  pl|          

In [ ]:
# Para auxiliar na análise também se criou uma nova coluna que contem a data só com mês e ano
df_spotify_eng = df_spotify_eng.withColumn("mes_ano", date_format(col("data"), "MM-yyyy"))
df_spotify_eng.show(15)

+----------+-------+----+--------------+--------------------+--------------------+-------------+-------------+----------+---------+------+---------------+-----------+-----------+-------------+-------+
|      data|posicao|pais|var_classifica|       nome_episodio|           descricao|nome_programa|      editora|duracao_ms|explicito|idioma|data_lancamento|forma_midia|epis_totais|faixa_duracao|mes_ano|
+----------+-------+----+--------------+--------------------+--------------------+-------------+-------------+----------+---------+------+---------------+-----------+-----------+-------------+-------+
|2025-10-30|    119|  pl|           NEW|Czy Polska Oddała...|W tym odcinku ser...| Dawid Sołtan| Dawid Sołtan| 3239658.0|    false|['pl']|     2025-10-28|      mixed|         85|        longo|10-2025|
|2025-10-31|    187|  pl|          DOWN|Czy Polska Oddała...|W tym odcinku ser...| Dawid Sołtan| Dawid Sołtan| 3239658.0|    false|['pl']|     2025-10-28|      mixed|         85|        longo|10-2